**STEP 1: Set Up Environment**

In [ ]:
# Display installation header with visual separator
print("="*70)
print(" INSTALLING PACKAGES")
print("="*70)
print("This will take 2-3 minutes. Please wait...")
print("(Dependency warnings are normal in Colab and can be ignored)\n")

# Install all required packages quietly with filtered output
# Packages installed:
# - langchain: Core LangChain framework
# - langchain-community: Community integrations for LangChain
# - langchain-google-genai: Google Gemini integration
# - sentence-transformers: For text embeddings
# - chromadb: Vector database for document storage
# - google-generativeai: Native Google Gemini SDK
# - google-search-results: For web search capabilities (optional)
# - pandas: Data manipulation and analysis
#
# The grep command filters out common dependency warnings to keep output clean
!pip install -q langchain langchain-community langchain-google-genai sentence-transformers chromadb google-generativeai google-search-results pandas 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true

# Display completion message
print("\n Installation complete!")
print("="*70)

In [ ]:
# Importing Libraries
# LangChain - Google Gemini Integration
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain - Core Components
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain

# Utility Libraries
import os
from google.colab import userdata

In [ ]:

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("✓ API Key configured successfully")
print("   (Key is stored securely and not displayed)")

**STEP 2: Initialize Language Model**

In [ ]:


# Initialize Gemini LLM with optimized parameters

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.7,
    convert_system_message_to_human=True
)

print("✓ Gemini LLM initialized successfully")
print(f"   Model: gemini-2.0-flash")
print(f"   Temperature: 0.7 (balanced creativity)")

**STEP 3: Design Prompt Templates**

In [ ]:
#  Chain 1 - Generate a question
# ============================================================================
# This chain creates a beginner-level question based on the provided topic.
# It's the first step in our sequential quiz generation pipeline.
# ============================================================================

# Define the prompt template for question generation
# This template takes a topic and generates an appropriate beginner question
question_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Generate a beginner-level question related to the topic: {topic}"
)



In [ ]:
#  Chain 2 - Answer the question
# ============================================================================
# This chain generates a clear, concise answer with explanation for the
# question created by Chain 1. It provides both the answer and reasoning.
# ============================================================================

# Define the prompt template for answer generation
# This template takes the generated question and produces an answer with explanation

answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="Provide a clear answer with a short explanation for the question: {question}"
)

print("✓ Chain 2 (Answer Generation) prompt template created")
print("   Purpose: Generate clear answers with explanations")

**STEP 4: Build Individual Chains**

In [ ]:
#  Create individual chains
# ============================================================================
# Instantiate the individual chains that will be connected in a sequence.
# Each chain wraps the LLM with its specific prompt template and output key.
# ============================================================================

# Chain 1: Question Generation Chain
# Takes 'topic' as input, outputs 'question'

question_chain = LLMChain(llm=llm, prompt=question_prompt, output_key="question")
answer_chain = LLMChain(llm=llm, prompt=answer_prompt, output_key="answer")

print("✓ Individual chains created successfully")
print("   Chain 1: topic → question")
print("   Chain 2: question → answer")

**STEP 5: Compose Sequential Chain**

In [ ]:
# ============================================================================
# Connect the individual chains into a sequential workflow where the output
# of Chain 1 (question) automatically becomes the input for Chain 2 (answer).
# This creates an end-to-end quiz generation pipeline.
# ============================================================================

# Create the sequential chain that orchestrates the entire workflow


chain = SequentialChain(
    chains=[question_chain, answer_chain],
    input_variables=["topic"],
    output_variables=["question", "answer"],
    verbose=True
)

print("✓ Sequential chain created successfully")
print("   Pipeline: topic → [Chain 1] → question → [Chain 2] → answer")
print("   Verbose mode: ENABLED (will show execution details)")

**STEP 6: Run Quiz Generator**

In [ ]:

# ============================================================================
# Run the complete sequential chain with user input and display the results.
# The pipeline will execute both chains automatically and return the final
# question-answer pair.
# ============================================================================

# Get user input for the quiz topic

topic = input("Enter a topic: ")

# Execute the sequential chain
# The chain will:
# 1. Generate a question from the topic (Chain 1)
# 2. Generate an answer for that question (Chain 2)
# 3. Return both outputs in a dictionary

response = chain.invoke({"topic": topic})


# ============================================================================
# DISPLAY RESULTS
# ============================================================================
# Present the generated quiz question and answer in a clean, readable format.
# ============================================================================

print("\n" + "="*70)
print(" GENERATED QUIZ")
print("="*70)

# Display the generated question
print("\n❓ Question:")
print(response["question"])

# Display the answer with explanation
print("\n Answer & Explanation:")
print(response["answer"])

print("\n" + "="*70)

